In [49]:
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client
from strands import Agent
from strands.tools.mcp import MCPClient
from strands.models import BedrockModel


GATEWAY_URL = "https://gateway-for-amazon-quick-wautpoplwm.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"
GATEWAY_URL = "https://gateway-with-search-iamjppxh4c.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"
GATEWAY_URL = "https://gateway-ehjdbglfzt.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"


mcp_client = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=GATEWAY_URL,
        aws_service="bedrock-agentcore",
    ),
)

# List available tools first
with mcp_client:
    tools = mcp_client.list_tools_sync()
    print("Available tools:")
    for t in tools:
        print(f"  - {t.tool_name}")

Available tools:
  - mcp-server___weather
  - rest-api___createSupportTicket
  - rest-api___getBalance
  - rest-api___getBetById
  - rest-api___getBetHistory
  - rest-api___getCustomer
  - rest-api___getCustomerBets
  - rest-api___getCustomerTickets
  - rest-api___getCustomerTransactions
  - rest-api___getEventsBySport
  - rest-api___getLiveEvents
  - rest-api___getOdds
  - rest-api___getPromotions
  - rest-api___getSupportTicket
  - rest-api___listSports
  - rest-api___listSupportTickets
  - rest-api___placeBet


In [46]:
model_id = "global.anthropic.claude-opus-4-6-v1"
model = BedrockModel(
    model_id=model_id,
)

# Create agent and use the tools
agent = Agent(model=model, tools=[mcp_client])
response = agent("weather in athens please")
print(f"\nAgent response: {response}")


Tool #1: mcp-server___weather
The weather in **Athens** is currently **very sunny** ☀️! Looks like a beautiful day there. Is there anything else you'd like to know?
Agent response: The weather in **Athens** is currently **very sunny** ☀️! Looks like a beautiful day there. Is there anything else you'd like to know?



In [47]:
# Create agent and use the tools
agent = Agent(model=model, tools=[mcp_client])
response = agent("I have a budget of $50. Find me the best value football bet right now — check the events, compare the odds, pick the one with the highest potential return")
print(f"\nAgent response: {response}")



I'll help you find the best value football bet! Let me start by finding football events and checking what's available.
Tool #1: api-gateway___listSports
Football has 24 active events. Let me pull up the football events now.
Tool #2: api-gateway___getEventsBySport
Great — 3 football events! Let me now check the odds for all of them simultaneously.
Tool #3: api-gateway___getOdds

Tool #4: api-gateway___getOdds

Tool #5: api-gateway___getOdds
I've got all the odds. Let me break down the analysis for you:

---

## 🏈 Football Odds Comparison

### Event 1: Arsenal vs Chelsea (Premier League)
| Selection | Odds | $50 Potential Return |
|-----------|------|---------------------|
| Home Win (Arsenal) | 2.10 | **$105** |
| Draw | 3.40 | **$170** |
| Away Win (Chelsea) | 3.20 | **$160** |

### Event 2: Barcelona vs Real Madrid (La Liga)
| Selection | Odds | $50 Potential Return |
|-----------|------|---------------------|
| Home Win (Barcelona) | 1.85 | **$92.50** |
| Draw | 3.60 | **$180** |
|

In [ ]:
response = agent("show me details of the proposed bet")

## Gateway Semantic Tool Search

AgentCore Gateway can index all tool metadata into a managed vector store and expose a built-in
`x_amz_bedrock_agentcore_search` MCP tool. You search with natural language, get back only the
relevant tools, and hand those to the agent — keeping the context window small and latency low.

> **Prerequisite:** The gateway must be created with `protocolConfiguration = {"mcp": {"searchType": "SEMANTIC"}}`.
> This cannot be added after creation.

In [ ]:
import json
from strands.tools.mcp.mcp_client import MCPClient, MCPAgentTool
from mcp.types import Tool as MCPTool

# Point to a gateway that has semantic search enabled
# GATEWAY_URL = "https://your-search-enabled-gateway.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"

search_client = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=GATEWAY_URL,
        aws_service="bedrock-agentcore",
    ),
)

### Step 1 — Search for tools with a natural language query

In [48]:
def extract_search_tools(result, top_n=3):
    """Extract tools from gateway search result, keeping only the top_n most relevant.
    
    The gateway returns up to 10 results ranked by relevance, but low-relevance
    tools are still included. Use top_n to keep only the best matches.
    """
    content = result.get("content", [])
    tools = None
    for item in content:
        if item.get("type") == "text" and item.get("text"):
            try:
                tools = json.loads(item["text"])["tools"]
                break
            except (json.JSONDecodeError, KeyError):
                pass
        if "structuredContent" in item:
            tools = item["structuredContent"]["tools"]
            break
    if tools is None and "structuredContent" in result:
        tools = result["structuredContent"]["tools"]
    if tools is None:
        raise ValueError(f"Unexpected search result format: {json.dumps(result, default=str)[:500]}")
    return tools[:top_n]


with search_client:
    result = search_client.call_tool_sync(
        tool_use_id="search-1",
        name="x_amz_bedrock_agentcore_search",
        arguments={"query": "assist betting and gaming users"},
    )
    # The search returns tool definitions ranked by relevance
    found_tools = json.loads(result["content"][0]["text"])["tools"]
    print(f"Found {len(found_tools)} relevant tools:")
    for t in found_tools:
        print(f"  - {t['name']}: {t['description'][:80]}...")

Found 10 relevant tools:
  - api-gateway___getCustomerBets: getCustomerBets...
  - api-gateway___placeBet: placeBet...
  - api-gateway___getBetHistory: getBetHistory...
  - api-gateway___getBetById: getBetById...
  - api-gateway___getOdds: getOdds...
  - api-gateway___listSports: listSports...
  - api-gateway___getEventsBySport: getEventsBySport...
  - api-gateway___listSupportTickets: listSupportTickets...
  - api-gateway___getCustomerTickets: getCustomerTickets...
  - api-gateway___getSupportTicket: getSupportTicket...


### Step 2 — Convert search results into agent tools and run the agent

In [ ]:
def search_to_agent_tools(search_results, client, top_n=5):
    """Convert gateway search results into Strands MCPAgentTool objects."""
    agent_tools = []
    for tool in search_results[:top_n]:
        mcp_tool = MCPTool(
            name=tool["name"],
            description=tool["description"],
            inputSchema=tool["inputSchema"],
        )
        agent_tools.append(MCPAgentTool(mcp_tool, client))
    return agent_tools


with search_client:
    # Search for betting tools
    result = search_client.call_tool_sync(
        tool_use_id="search-2",
        name="x_amz_bedrock_agentcore_search",
        arguments={"query": "place a bet, check odds, view balance"},
    )
    found_tools = json.loads(result["content"][0]["text"])["tools"]

    # Build a focused agent with only the relevant tools
    focused_agent = Agent(
        model=model,
        tools=search_to_agent_tools(found_tools, search_client, top_n=5),
        system_prompt="You are a betting assistant for a sports betting platform.",
    )
    response = focused_agent("What are the odds for the Arsenal vs Chelsea match?")
    print(response.message["content"][0]["text"])

### Step 3 — Different query, different tools, same gateway

In [ ]:
with search_client:
    # Now search for customer support tools instead
    result = search_client.call_tool_sync(
        tool_use_id="search-3",
        name="x_amz_bedrock_agentcore_search",
        arguments={"query": "weather tools"},
    )
    support_tools = json.loads(result["content"][0]["text"])["tools"]
    print(f"Found {len(support_tools)} support tools:")
    for t in support_tools:
        print(f"  - {t['name']}")

    # Build a support agent with only those tools
    support_agent = Agent(
        model=model,
        tools=search_to_agent_tools(support_tools, search_client, top_n=5),
        system_prompt="You are a weather support agent.",
    )
    response = support_agent("Show me the transaction history for customer cust-003")
    print(response.message["content"][0]["text"])